# **Machine Learning Part 4: Grid Search CV**

In [1]:
# import the librareis

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# our data

df = sns.load_dataset('iris')
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
# i want to check how many species are there
# because i have to predict these species
df['species'].unique()

<StringArray>
['setosa', 'versicolor', 'virginica']
Length: 3, dtype: str

In [ ]:
# do the train test split before the prediciotn 
from sklearn.model_selection import train_test_split

# make the x varibale and y
X = df.drop('species',axis = 1)
y = df['species']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

+ use knn model

In [7]:
# now i will use KNN model and use it

from sklearn.neighbors import KNeighborsClassifier # import the model

model_knn = KNeighborsClassifier(n_neighbors=13) # initialze the object

model_knn.fit(X_train,y_train) # train the model

model_knn.score(X_test,y_test) # test the model and get the accuracy score



1.0

+ use svm model

In [10]:
# Import the Support Vector Classifier (SVC) from scikit-learn
from sklearn.svm import SVC

# Initialize the SVM model
# gamma='auto' means gamma = 1 / n_features
# It controls how far the influence of a single training example reaches
model_svm = SVC(gamma='auto') # HERE C AND KERNEL ARE HIDDEN AND AT BYDEFAULT

# Train (fit) the SVM model using the training data
# X_train = input features for training
# y_train = target labels for training
model_svm.fit(X_train, y_train)

# Evaluate the model on the test data
# X_test = input features for testing
# y_test = true labels for testing
# .score() returns the accuracy of the model (fraction of correct predictions)
accuracy = model_svm.score(X_test, y_test)

# Print the accuracy score
print("Model Accuracy:", accuracy)


Model Accuracy: 1.0


**USE THE GRID SEARCH**

In [20]:

# Import GridSearchCV for hyperparameter tuning
from sklearn.model_selection import GridSearchCV

# Perform grid search on the SVM model
# Parameters to tune:
#   - 'C': Regularization parameter (higher values = less regularization)
#   - 'kernel': Type of kernel function ('rbf' or 'linear')
# cv=5 means 5-fold cross-validation will be used
# return_train_score=False means only test scores will be stored
classifier = GridSearchCV(
    estimator=model_svm,                # Base model (SVM) which model you want to use
    param_grid={
        'C': [1, 10, 20, 30],           # Different values of regularization parameter
        'kernel': ['rbf', 'linear']     # Try both RBF and Linear kernels
    },
    cv=5,                               # 5-fold cross-validation
    return_train_score=False            # Do not store training scores
)

# Fit the classifier on the entire dataset (X = features, y = labels)
classifier.fit(X, y) # here the model will be made

# Convert the cross-validation results into a DataFrame for easy analysis
results = pd.DataFrame(classifier.cv_results_)

# Display the results table
results


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002204,0.000541,0.001659,0.000190,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.002078,0.000240,0.001533,0.000090,1,linear,"{'C': 1, 'kernel': 'linear'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
2,0.003143,0.000193,0.002427,0.000651,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
3,0.002663,0.000719,0.002042,0.000589,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.966667,1.0,0.973333,0.038873,4
4,0.002437,0.000613,0.001836,0.000420,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.0,0.900000,0.966667,1.0,0.966667,0.036515,5
5,0.001905,0.000245,0.001469,0.000076,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,6
6,0.001874,0.000171,0.001646,0.000333,30,rbf,"{'C': 30, 'kernel': 'rbf'}",0.966667,1.0,0.900000,0.933333,1.0,0.960000,0.038873,7
7,0.001825,0.000259,0.001497,0.000233,30,linear,"{'C': 30, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.900000,1.0,0.960000,0.048990,7


In [21]:

results[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667
6,30,rbf,0.960000
7,30,linear,0.960000


**USE RANDOM SEARCH CV**

In [18]:
# Import RandomizedSearchCV for hyperparameter tuning with random sampling
from sklearn.model_selection import RandomizedSearchCV

# Perform randomized search on the SVM model
# Instead of trying all combinations (like GridSearchCV),
# RandomizedSearchCV randomly samples a fixed number of parameter settings.
classifier_r = RandomizedSearchCV(
    estimator=model_svm,                # Base model (SVM)
    param_distributions={               # Dictionary of parameters to sample from
        'C': [1, 10, 20, 30],           # Regularization parameter values
        'kernel': ['rbf', 'linear']     # Kernel types to try
    },
    n_iter=4,                           # Number of random combinations to test
    cv=5,                               # 5-fold cross-validation
    return_train_score=False            # Do not store training scores
)

# Fit the randomized search classifier on the dataset
# X = input features, y = target labels
classifier_r.fit(X, y)

# Convert the cross-validation results into a DataFrame for easy analysis
results = pd.DataFrame(classifier_r.cv_results_)

# Display the results table (shows parameters tried and their scores)
results


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kernel,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002371,0.000537,0.001537,0.000122,linear,30,"{'kernel': 'linear', 'C': 30}",1.000000,1.0,0.900000,0.900000,1.0,0.960000,0.048990,4
1,0.002284,0.000254,0.001542,0.000103,linear,1,"{'kernel': 'linear', 'C': 1}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
2,0.001843,0.000086,0.001463,0.000076,rbf,1,"{'kernel': 'rbf', 'C': 1}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
3,0.001755,0.000087,0.001412,0.000061,linear,20,"{'kernel': 'linear', 'C': 20}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,3


In [19]:

results[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,30,linear,0.960000
1,1,linear,0.980000
2,1,rbf,0.980000
3,20,linear,0.966667
